# Induction Evals — One-Hop (Range Query)

> **⚠️ `noise_intens` must be re-run (2026-08-02).** The noise arm is a *length control*, and it used to be sized in CHARACTERS with random alphanumeric filler — which tokenizes so much worse than structured text that the control ran **1.4–1.6× longer in tokens** than the extensional arm it was controlling for. It now pads with **whitespace, matched to an exact TOKEN count** against each rendered extensional prompt, using the tokenizer of the model under test (so `make_quizzes` takes `(seed, model)`, and only this arm differs per model). Every `*_noise_intens/` results directory was deleted in that change — the old data is in git history, but it is not comparable to what this notebook now generates, and `power_analysis.py` will stop with a clear message until the arm is re-collected. `intens`/`extens`/`zero` are untouched and still resume-skip.

This notebook is the **one-hop** sibling of `induction_eval.ipynb`: instead of direct-succession pairs, each question asks whether a single sampled year could belong to a colour, so the intensional and extensional representations each require exactly **one hop** (one boundary check vs. one year-keyed lookup).

**Flow:** generate this experiment's quizzes → build an `InductionExperiment` → provision ONE EC2 spot instance (shared with `induction_eval.ipynb` via the same experiment tag if run back-to-back) → run each archetype in turn via `EXPERIMENT.run(model, ...)` (each swaps the shared instance's vLLM to that model) → `EXPERIMENT.summarize(model)` → `EXPERIMENT.teardown()` at the end. All lifecycle/harness plumbing lives in `smolbench.induction.experiment.InductionExperiment`; this notebook only supplies its experiment-specific config (results directory, archetype tags, quiz factory, replicate count, EC2 state-file namespace, and the `one_hop_` results prefix below).

**Replication design:** each (archetype, info type) condition runs **R = 30 replicates** of the same one-hop True/False quiz, each under a fresh `ChromaticIntervalsConfig` seed (fresh interval history, colour labels, and False-year sampling, with that seed reused as the per-request decoding seed). `BASE_SEED = 1776`; replicate `r`'s seed is `BASE_SEED + r` -- a replicate is never a bigger quiz, it is the identical quiz regenerated fresh under a new seed. The unit of observation is the **quiz**: its ~120 questions share one context and are dominated by a near-global True/False response bias, so they are not iid. A replicate re-runs the *whole* quiz under a new seed -- it never enlarges a single quiz. R = 30 mirrors `induction_eval.ipynb` / `notebooks/periodic`.

**Resume semantics:** every archetype cell is idempotent -- `EXPERIMENT.run()` serializes each (archetype, info type, seed) replicate to `results/one_hop_{archetype}_{info}/rep_{seed}.yaml` immediately after it is graded (the `one_hop_` prefix namespaces this experiment's replicate dirs so they never collide with `induction_eval.ipynb`'s, which share the same `results/` tree -- see `smolbench/evals/replicates.py`), so replicates already on disk are skipped and an interrupted section resumes exactly where it left off. All 30 replicates (seeds 1776–1805) are generated fresh under the current olmo/granite trio; the earlier single-seed run used a now-undeployable trio and was discarded, so there is no migrated `rep_1776.yaml` here.

**Cost warning:** provisioning spins up one EC2 spot instance for the whole notebook (~$30-45/h for the p5e/p5 family -- mind the meter); provision right before running the sections below, not hours ahead. An on-instance watchdog terminates the box after ~30 idle minutes without traffic, and an absolute max-lifetime backstop exists regardless -- see the Teardown section at the bottom for the graceful path.

**keys.env:** the first executable statement below MUST stay `load_dotenv(...)`. `smolbench.evals.ec2` captures its `EC2_*` config (including this experiment's `EC2_EXPERIMENT_TAG` and instance-type allowlist) from `os.environ` at IMPORT time, so keys.env's values must land before that module is ever imported; `InductionExperiment` imports it lazily inside each method for exactly this reason (see `smolbench/induction/experiment.py`'s module docstring).

In [ ]:
"""Generates the one-hop range quiz that all models are evaluated on."""

import string

import logging

from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
load_dotenv("./keys.env", verbose=True)

from typing import Dict, Tuple

from smolbench.induction.chromatic import (
    ChromaticIntervalsConfig,
    Prompter,
    get_random_exclusive_quiz,
    one_hop_year_query_gen,
)
from smolbench.evals.tokenization import for_model

# Inference runs on a self-provisioned EC2 spot instance serving vLLM
# (smolbench/evals/ec2.py), exactly like notebooks/periodic. keys.env
# (gitignored) selects the provider and this experiment's static EC2 config:
# INFERENCE_PROVIDER=ec2, EC2_EXPERIMENT_TAG=chromatic-induction (isolates
# this experiment's instance from notebooks/periodic), and
# EC2_INSTANCE_TYPES=p5e.48xlarge,p5.48xlarge (8-GPU H200/H100 at tp=8: the
# 4x L40S g6e boxes the trio first ran on were decode-bandwidth bound; see
# the EC2_DEPLOY_SPECS comment in smolbench/evals/ec2.py). InductionExperiment
# (below) applies INFERENCE_PROVIDER and this experiment's private EC2 state
# file (".ec2_state_chromatic.json", kept separate from periodic's
# .ec2_state.json) at call time -- see smolbench/induction/experiment.py.

# Per-archetype model names: keys of EC2_DEPLOY_SPECS in
# smolbench/evals/ec2.py (each key is also vLLM's --served-model-name). An
# ungated ~32B English-centric trio -- no HF account/login/token needed.
DENSE_MODEL = "olmo-3.1-32b-instruct"  # allenai/Olmo-3.1-32B-Instruct (32B dense, non-reasoning)
COT_MODEL = "olmo-3.1-32b-think"       # allenai/Olmo-3.1-32B-Think (32B dense, always reasons)
MOE_MODEL = "granite-4.0-h-small"      # ibm-granite/granite-4.0-h-small (MoE 32B/~9B active, non-reasoning)
# Olmo-3.1-32B-Think's chat template force-opens <think>, so it reasons
# unconditionally; give the completion enough budget to finish thinking AND
# emit the answer. query() splits the think block into the reasoning channel
# client-side, so scoring sees only the answer.
COT_EXTRA_ARGS = {"max_completion_tokens": 8192}
# CoT runs must let the LONGEST chain finish on attempt 1: a tight client
# timeout + high concurrency censors long-CoT requests (they time out, retry,
# and survive only via the scheduler lottery -> the measured CoT-length
# distribution is cut off at the top, and the scored output stops being
# seed-deterministic). Low parallelism gives each chain enough GPU to finish
# well under a generous timeout, uniformly.
COT_MAX_PARALLEL = 16       # p5e/p5 (8-GPU H200/H100) has huge KV headroom + ~5-7x faster decode, so feed it wider
COT_REQUEST_TIMEOUT = 1200  # seconds: covers an 8192-token chain even at low decode tok/s

template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists the people who were $role and"
    " the years they were $role:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "During year $year, could $color have headed the $parade parade?"
)

extens_template = string.Template(
    "You are a Boolean classifier.\n"
    "\n"
    "Task: determine whether the statement in the Question is logically "
    "possible given the Context.\n"
    "\n"
    "Output format:\n"
    "Return exactly one of these two strings and nothing else:\n"
    "True\n"
    "False\n"
    "\n"
    "Do not output any explanation, punctuation, quotes, labels, code fences, "
    "or extra whitespace."
    "Stop immediately after writing True or False."
    "\n"
    "Context:\n"
    "There is a ceremonial role called the $role, whose job it is to"
    " head the $parade parade. No one else besides the $role is able to head"
    " the $parade parade. The following lists each year and who was $role"
    " that year:\n"
    "$positive_info\n"
    "\n"
    "Question:\n"
    "During year $year, could $color have headed the $parade parade?"
)

# The one-hop query generator is the module builtin
# smolbench.induction.chromatic.one_hop_year_query_gen (imported above), for
# the same reason its succession sibling lives there: one importable, tested
# definition of the task.
QUERY_GEN = one_hop_year_query_gen

# --- Replication setup ---------------------------------------------------
# See the intro cell for the full replicate/power-analysis rationale (quiz
# as the unit of observation, R=30, the BASE_SEED convention). R and the
# per-replicate seed list are now InductionExperiment constructor args
# (cell below); BASE_SEED and INFO_TYPES stay here because make_quizzes
# (below) and the Prompt Validation aliases reference them directly.
BASE_SEED: int = 1776  # base seed; all R replicates generated fresh (no migrated rep_1776)
INFO_TYPES: Tuple[str, ...] = ("intens", "extens", "noise_intens")


def make_quizzes(seed: int, model: str) -> Dict[str, tuple]:
    """Generates one replicate's three info-type quizzes, keyed by info type.

    Chromatic quizzes are large -- every prompt embeds the full interval
    history (the extens listing alone is ~3000 lines) across ~120 questions
    -- so replicates are generated on demand per seed inside EXPERIMENT.run,
    not all precomputed up front, to keep notebook memory bounded.

    Takes the model as well as the seed because ``noise_intens`` is a TOKEN-
    length control: it is padded with whitespace until its prompt has exactly
    as many tokens as the matching extensional prompt, measured with that
    model's own tokenizer. The other conditions are model-independent and stay
    byte-identical across models.
    """
    return dict(
        zip(
            INFO_TYPES,
            get_random_exclusive_quiz(
                ChromaticIntervalsConfig(
                    n=int(12 * 250),
                    intervals=250 // 4,
                    colors=45,
                    seed=seed,
                ),
                Prompter(
                    template,
                    {
                        "role": "Twislax",
                        "parade": "Gildane",
                    },
                    QUERY_GEN,
                    extens_template,
                ),
                tokenizer=for_model(model),
            ),
        )
    )


# First-replicate aliases for the Prompt Validation cells below (the only
# replicate generated eagerly; the rest are built per seed when run).
# (Only noise_intens depends on which model is named: it is token-matched
# with that model's own tokenizer. intens/extens are identical for all.)
_base_quizzes: Dict[str, tuple] = make_quizzes(BASE_SEED, DENSE_MODEL)
intens_quiz, extens_quiz, noise_intens_quiz = (
    _base_quizzes["intens"],
    _base_quizzes["extens"],
    _base_quizzes["noise_intens"],
)


In [ ]:
# Builds this notebook's InductionExperiment: bundles the replicate harness
# (results layout + quiz factory -- results/{prefix}{tag}_{info}/rep_{seed}.yaml,
# serialized immediately after grading so a spot interruption loses at most
# one replicate, and a seed's outstanding info types are pooled into one
# evaluate() call to keep the GPU saturated) with the EC2 spot-instance
# lifecycle. See smolbench/evals/replicates.py and
# smolbench/induction/experiment.py for the full mechanics.
from smolbench.induction.experiment import InductionExperiment

EXPERIMENT = InductionExperiment(
    notebook_dir="chromatic",
    archetype_tags={DENSE_MODEL: "decode", COT_MODEL: "cot", MOE_MODEL: "moe"},
    make_quizzes=make_quizzes,
    n_replicates=30,
    base_seed=BASE_SEED,
    state_file=".ec2_state_chromatic.json",
    # one_hop shares the results/ tree with induction_eval.ipynb; the prefix
    # namespaces its replicate dirs as results/one_hop_{tag}_{info}/ so the
    # two experiments never collide (power_analysis.py reads only the
    # unprefixed dirs).
    prefix="one_hop_",
)


In [ ]:
# Provisions this experiment's EC2 spot instance (idempotent -- reattaches
# via .ec2_state_chromatic.json / the smolbench:experiment tag if one is
# already up, so it's safe after kernel restarts; induction_eval.ipynb
# shares the same experiment tag, so both notebooks reuse one instance if
# run back-to-back). See the intro cell for the cost/safety-net notes and
# InductionExperiment.provision() for details.
state = EXPERIMENT.provision()


## Prompt Validation

In [ ]:
print(intens_quiz[0].prompt)

In [ ]:
print(extens_quiz[0].prompt)

In [ ]:
print(noise_intens_quiz[0].prompt)

## Decoder-Only Model
This section tests `olmo-3.1-32b-instruct` (allenai/Olmo-3.1-32B-Instruct) -- a 32B dense, non-reasoning model representing the "decode" archetype. The run cell below swaps the shared instance's vLLM container to this model, which takes several minutes while the checkpoint downloads/loads (fast on rerun -- the instance's HF cache is warm).

In [ ]:
# EXPERIMENT.run() swaps the shared instance's vLLM to DENSE_MODEL (a
# container swap; the first serve waits on the checkpoint download, reruns
# hit the instance's cache) and runs all outstanding replicates x 3 info
# types; safe to rerun after an interruption -- finished replicates are
# skipped. The instance keeps running for the next section afterward.
EXPERIMENT.run(DENSE_MODEL)

In [ ]:
# Aggregate results over all serialized decode replicates.
EXPERIMENT.summarize(DENSE_MODEL)

## CoT Model
This section tests `olmo-3.1-32b-think` (allenai/Olmo-3.1-32B-Think) -- a 32B dense model representing the "cot" archetype; its chat template force-opens `<think>`, so it reasons unconditionally, and `query()` splits the think block into the reasoning channel client-side so scoring sees only the answer. The run cell swaps the shared instance's vLLM to this model, applies `max_completion_tokens=8192` so a full chain has room to close before answering, and widens `max_parallel` to 16 with a 1200s `request_timeout` -- this box's KV headroom and faster decode support more concurrency than earlier deployments, and the long timeout ensures even the longest chain finishes on attempt 1 rather than being censored into a non-deterministic, top-truncated retry.

In [ ]:
# EXPERIMENT.run() swaps the instance's vLLM to COT_MODEL. Olmo-3.1-32B-Think
# reasons unconditionally (its chat template force-opens <think>); query()
# splits the think block into the reasoning channel client-side, so scoring
# sees only the answer. The wide max_parallel + long request_timeout are
# applied to every info type and replicate so the longest chain finishes on
# attempt 1 (a tight timeout censors long-CoT requests -> non-deterministic,
# top-truncated output).
EXPERIMENT.run(
    COT_MODEL,
    extra_args=COT_EXTRA_ARGS,
    max_parallel=COT_MAX_PARALLEL,
    request_timeout=COT_REQUEST_TIMEOUT,
)

In [ ]:
# Aggregate results over all serialized cot replicates.
EXPERIMENT.summarize(COT_MODEL)

In [ ]:
# Reasoning-chain length analysis: word counts from the cached CoT YAMLs
# (~1.3 tokens/word for these tokenizers). A top-truncated distribution here
# flags a too-tight CoT timeout (see the request_timeout note above).
EXPERIMENT.cot_chain_lengths()


## MoE Model
This section tests `granite-4.0-h-small` (ibm-granite/granite-4.0-h-small) -- a mixture-of-experts checkpoint (~32B total / ~9B active parameters per token, non-reasoning) representing the "moe" archetype. The run cell below swaps the shared instance's vLLM to this model.

In [ ]:
# EXPERIMENT.run() swaps the instance's vLLM to MOE_MODEL.
EXPERIMENT.run(MOE_MODEL)

In [ ]:
# Aggregate results over all serialized moe replicates.
EXPERIMENT.summarize(MOE_MODEL)

# Teardown
Gracefully shuts the experiment's EC2 spot instance down. If this cell is forgotten, the instance still self-terminates: an on-instance watchdog fires after `EC2_IDLE_TIMEOUT_MIN` (default 30) minutes without inference traffic or control-agent activity, and an absolute `EC2_MAX_LIFETIME_MIN` (default 24h) `shutdown -h` backstop is scheduled at boot. Both run on the instance itself, so they work even if this notebook's kernel is gone.

In [ ]:
# Terminates the spot instance (and its EBS volume) and clears
# .ec2_state_chromatic.json. Also works after a kernel restart or a lost
# state file: it falls back to the smolbench:experiment instance tag.
EXPERIMENT.teardown()